# 03. 메시지(Messages) — LangGraph 대화의 기본 단위

> LangGraph 상태의 핵심은 결국 **메시지 리스트**예요. System/Human/AI/Tool 4가지 타입과 메타데이터, `add_messages` 리듀서의 동작을 명확히 이해하고 갑니다.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage` 4가지 메시지 타입의 역할과 사용 시점을 설명할 수 있어요
2. `name`, `id` 메타데이터를 활용하여 다중 사용자 환경에서 메시지를 추적할 수 있어요
3. 딕셔너리 형식(`{"role": ..., "content": ...}`)으로 메시지를 구성할 수 있어요
4. `ToolMessage`의 `tool_call_id` 매칭 규칙을 이해하고 도구 호출 전체 흐름을 구현할 수 있어요
5. Provider 네이티브 형식과 표준 `content_blocks`를 활용하여 멀티모달 메시지를 구성할 수 있어요

## 사전 지식

- 이전 노트북: `02-Models.ipynb` — `init_chat_model`, `invoke/stream/batch`, Tool Calling 기초
- LangChain V0의 메시지 기본 개념


## 메시지란 무엇인가요?

메시지(Message)는 LangChain에서 LLM과 주고받는 대화의 기본 단위예요. 에이전트의 모든 상호작용은 메시지 리스트로 구성되고, LangGraph의 상태(State)에 저장돼요.

각 메시지는 다음 세 가지 핵심 요소로 구성돼요:

| 요소 | 역할 | 예시 |
|------|------|------|
| **역할(Role)** | 메시지 발신자 구분 | `system`, `user`, `assistant`, `tool` |
| **콘텐츠(Content)** | 실제 메시지 내용 | 텍스트, 이미지, 오디오 등 |
| **메타데이터(Metadata)** | 추가 정보 | `name`, `id`, 토큰 사용량, 도구 호출 정보 |

```mermaid
flowchart TD
    A["사용자 입력"]:::input --> B["HumanMessage"]:::input
    C["시스템 설정"]:::process --> D["SystemMessage"]:::process
    B --> E["LLM 모델"]:::process
    D --> E
    E --> F["AIMessage"]:::output
    F --> G{"도구 호출?"}:::process
    G -->|"Yes: tool_calls"| H["도구 실행"]:::storage
    H --> I["ToolMessage"]:::storage
    I --> E
    G -->|"No: content"| J["최종 응답"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

### LangGraph에서 메시지의 위치

LangGraph 에이전트의 상태에는 항상 `messages` 필드가 있어요. 각 노드(Node)는 이 리스트에서 메시지를 읽고 새로운 메시지를 추가해요.

> 🔑 **핵심 개념**: LangGraph에서 `add_messages` reducer는 메시지 리스트를 자동으로 관리해요. 같은 `id`를 가진 메시지는 업데이트되고, 새로운 메시지는 추가돼요. 다음 노트북 `04-StateGraph-Basics.ipynb`에서 자세히 다룰 거예요.


## 환경 설정


In [1]:
# ---------------------------------------------------
# 환경 설정
# ---------------------------------------------------
# dotenv: .env 파일에서 API 키를 로드해요
from dotenv import load_dotenv
import os

# .env 파일의 환경 변수를 로드해요
load_dotenv(override=True)

# LangSmith 추적 설정 (선택 사항)
# LangSmith에서 메시지 흐름을 시각적으로 디버깅할 수 있어요
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-Part03-Messages"


True

## 1. 메시지 기본 사용법

메시지 객체를 생성하고 모델에 리스트로 전달하는 가장 기본적인 패턴이에요. 메시지 리스트를 통해 대화 컨텍스트를 유지할 수 있어요.

### 메시지 Import 경로 (V1)

LangChain V1에서는 모든 메시지를 `langchain.messages`에서 가져와요.

| V1 (올바른 방법) | 이전 방식 (피해야 할 방법) |
|-----------------|---------------------------|
| `from langchain.messages import HumanMessage` | `from langchain_core.messages import HumanMessage` |


In [2]:
# ---------------------------------------------------
# V1 메시지 Import 및 모델 초기화
# ---------------------------------------------------
# V1 통합 경로: langchain.messages (langchain_core 대신)
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain.chat_models import init_chat_model

# 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# 다른 모델: "anthropic:claude-sonnet-4-5", "google_genai:gemini-2.0-flash"
model = init_chat_model("openai:gpt-4o-mini")

print(f"모델 초기화 완료: {type(model).__name__}")

모델 초기화 완료: ChatOpenAI


In [3]:
# ---------------------------------------------------
# 메시지 객체를 사용한 기본 모델 호출
# ---------------------------------------------------
# SystemMessage: 모델의 역할과 동작 방식을 지시해요
system_msg = SystemMessage("당신은 친절한 AI 어시스턴트입니다. 한국어로 답변해주세요.")

# HumanMessage: 사용자의 질문이나 요청을 담아요
human_msg = HumanMessage("안녕하세요. LangGraph란 무엇인가요?")

# 메시지 리스트로 모델 호출
messages = [system_msg, human_msg]
response = model.invoke(messages)  # AIMessage 객체를 반환해요

# 응답 확인
print(f"응답 타입: {type(response).__name__}")
print(f"응답 내용: {response.content}")

응답 타입: AIMessage
응답 내용: 안녕하세요! LangGraph는 일반적으로 언어 모델이나 자연어 처리와 관련된 그래프 구조를 의미할 수 있습니다. 하지만 구체적으로 LangGraph라는 이름의 프로젝트나 제품에 대한 정보가 필요합니다. 

일반적으로, 이런 개념은 언어 데이터의 관계를 시각화하고 분석하는 데 사용되며, 언어 간의 연관성을 이해하는 데 도움을 줄 수 있습니다. 

해당 용어가 특정한 기술, 프로젝트, 또는 소프트웨어를 가리키는 것이라면, 추가적인 정보나 맥락을 제공해 주시면 더 정확한 답변을 드릴 수 있습니다.


### 딕셔너리 형식으로 메시지 구성하기

메시지 객체 대신 딕셔너리로도 메시지를 지정할 수 있어요. `role` 키에는 역할을, `content` 키에는 내용을 넣어요. JSON 직렬화가 필요하거나 외부 API 연동 시 편리해요.

| 딕셔너리 `role` | 메시지 객체 |
|---|---|
| `system` | `SystemMessage` |
| `user` | `HumanMessage` |
| `assistant` | `AIMessage` |
| `tool` | `ToolMessage` |


In [4]:
# ---------------------------------------------------
# 딕셔너리 형식으로 메시지 구성
# ---------------------------------------------------
# 딕셔너리의 role 키: "system", "user", "assistant" 사용
# ("human" 대신 "user"를 사용해야 해요)
messages_dict = [
    {"role": "system", "content": "당신은 친절한 어시스턴트입니다."},
    {"role": "user", "content": "대한민국의 수도는?"},
    {"role": "assistant", "content": "대한민국의 수도는 서울입니다."},
    {"role": "user", "content": "영어로 작성해줘."},
]

# 딕셔너리 형식과 객체 형식은 동일하게 동작해요
response_dict = model.invoke(messages_dict)
print(f"응답: {response_dict.content}")
print()

응답: The capital of South Korea is Seoul.



## 2. 4가지 메시지 타입 상세 가이드

LangChain V1은 4가지 핵심 메시지 타입을 제공해요. 각 타입은 대화에서 명확한 역할을 담당해요.

| 메시지 타입 | 역할 | 대화에서 위치 |
|------------|------|---------------|
| `SystemMessage` | 모델 동작 방식과 역할 정의 | 대화 시작 부분 |
| `HumanMessage` | 사용자 입력 (텍스트 + 멀티모달) | 사용자 차례마다 |
| `AIMessage` | 모델 생성 응답 (텍스트 + 도구 호출) | 모델 응답마다 |
| `ToolMessage` | 도구 실행 결과 전달 | 도구 호출 직후 |


### 2-1. SystemMessage — 모델 행동 지침

`SystemMessage`는 모델의 페르소나, 응답 스타일, 제약 조건 등을 정의해요. 효과적인 시스템 메시지는 모델이 일관된 방식으로 동작하도록 만들어요.

> 🔑 **핵심 개념**: 시스템 메시지는 대화 전체에 영향을 미치는 "설정" 이에요. 역할(`role`), 제약(`constraint`), 형식(`format`) 세 가지 요소를 포함하는 것이 좋아요.


In [5]:
# ---------------------------------------------------
# SystemMessage: 모델 역할과 동작 방식 지정
# ---------------------------------------------------
from langchain.messages import SystemMessage, HumanMessage

# 역할(role) + 제약(constraint) + 형식(format)을 포함한 시스템 메시지
system_msg = SystemMessage(
    "You are a helpful Python coding assistant. "
    "Always provide working code examples. "
    "Use Korean for explanations but English for code."
)

# 시스템 메시지를 포함한 대화 구성
messages = [
    system_msg,
    HumanMessage("리스트에서 중복을 제거하는 코드를 보여줘."),
]

response = model.invoke(messages)
print(response.content)

리스트에서 중복된 요소를 제거하는 방법은 여러 가지가 있습니다. 여기서는 `set`을 사용하여 중복을 제거하는 간단한 예제를 보여드리겠습니다. 

```python
# 중복된 요소가 있는 리스트
original_list = [1, 2, 2, 3, 4, 4, 5, 5, 6]

# set을 사용하여 중복 제거
unique_list = list(set(original_list))

print(unique_list)
```

위 코드는 `original_list`의 중복된 요소를 제거하고, 결과를 `unique_list`에 저장하여 출력합니다. `set`은 중복을 허용하지 않기 때문에 간단하게 사용할 수 있습니다. 그러나 원래 리스트의 순서가 중요하다면 다른 방법을 사용할 수 있습니다.

리스트의 순서를 유지하면서 중복을 제거하는 방법도 있습니다:

```python
# 중복된 요소가 있는 리스트
original_list = [1, 2, 2, 3, 4, 4, 5, 5, 6]

# 순서를 유지하며 중복 제거
unique_list = []
for item in original_list:
    if item not in unique_list:
        unique_list.append(item)

print(unique_list)
```

이 방법은 원래 리스트의 순서를 유지하면서 중복을 제거합니다. 사용하실 방식에 따라 선택하시면 됩니다.


### 2-2. HumanMessage — 사용자 입력

`HumanMessage`는 사용자가 보내는 모든 입력을 담아요. 텍스트뿐 아니라 이미지, 오디오 등 멀티모달 콘텐츠도 포함할 수 있어요.


In [6]:
# ---------------------------------------------------
# HumanMessage: 기본 생성 및 메타데이터 추가
# ---------------------------------------------------
from langchain.messages import HumanMessage

# 기본 텍스트 메시지
basic_human = HumanMessage("안녕하세요?")
# 기본 HumanMessage:
print(f"  content: {basic_human.content}")
print(f"  type: {basic_human.type}")
print()

# 메타데이터를 포함한 메시지
# name: 다중 사용자 환경에서 발신자를 식별해요
# id: 메시지를 고유하게 식별해요 (add_messages에서 업데이트 기준)
human_with_meta = HumanMessage(
    content="LangGraph가 궁금해요!",
    name="student_alex",  # 발신자 이름 (선택 사항)
    id="msg-001",          # 고유 식별자 (선택 사항)
)

# 메타데이터 포함 HumanMessage:
print(f"  content: {human_with_meta.content}")
print(f"  name: {human_with_meta.name}")
print(f"  id: {human_with_meta.id}")

  content: 안녕하세요?
  type: human

  content: LangGraph가 궁금해요!
  name: student_alex
  id: msg-001


In [7]:
# ---------------------------------------------------
# 메타데이터가 포함된 메시지로 모델 호출
# ---------------------------------------------------
# name 필드는 모델에게 발신자 정보를 전달해요
response = model.invoke([human_with_meta])
print(f"응답: {response.content}")

# 반환된 AIMessage 구조 확인
print()
print(f"응답 타입: {type(response).__name__}")
print(f"응답 id: {response.id}")

응답: LangGraph는 자연어 처리(NLP)와 언어 모델링 분야에서 사용되는 비교적 새로운 개념이나 도구일 수 있습니다. 그러나 구체적인 정보는 제가 알고 있는 데이터 기준인 2023년 10월까지의 정보에는 포함되어 있지 않습니다. 

일반적으로 언어 모델들은 텍스트 데이터를 기반으로 언어의 패턴을 학습하고, 이를 통해 텍스트 생성, 문서 요약, 번역 등 다양한 작업을 수행할 수 있습니다. LangGraph도 비슷한 목적이나 방법론을 가지고 있을 가능성이 있습니다. 

더 구체적인 정보를 원하시면, LangGraph에 대한 출처나 구체적인 질문을 제공해 주신다면 보다 상세히 도와드릴 수 있을 것 같습니다!

응답 타입: AIMessage
응답 id: lc_run--019e1b0f-97c9-7940-8749-f4d0e100432a-0


### 2-3. AIMessage — 모델 응답

`AIMessage`는 모델이 생성한 응답이에요. 텍스트 응답 외에도 도구 호출(`tool_calls`) 정보, 토큰 사용량(`usage_metadata`), 제공자 메타데이터를 포함해요.


In [8]:
# ---------------------------------------------------
# AIMessage 구조 상세 분석
# ---------------------------------------------------
from langchain.messages import HumanMessage

# 일반 텍스트 응답
response = model.invoke([HumanMessage("파이썬이란?")])

# === AIMessage 구조 ===
print(f"[type]           : {response.type}")
print(f"[id]             : {response.id}")
print(f"[content]        : {response.content[:80]}...")  # 너무 길면 일부만 출력
print(f"[tool_calls]     : {response.tool_calls}")  # 도구 호출 없으면 빈 리스트
print(f"[usage_metadata] : {response.usage_metadata}")
print(f"[model_name]     : {response.response_metadata.get('model_name', 'N/A')}")

[type]           : ai
[id]             : lc_run--019e1b10-cf20-75c2-a8da-5f7c920b70c1-0
[content]        : 파이썬(Python)은 고급 프로그래밍 언어로, 1991년 귀도 반 로썸(Guido van Rossum)에 의해 처음 발표되었습니다. 간결하고 ...
[tool_calls]     : []
[usage_metadata] : {'input_tokens': 14, 'output_tokens': 343, 'total_tokens': 357, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
[model_name]     : gpt-4o-mini-2024-07-18


In [9]:
# ---------------------------------------------------
# 다중 턴 대화: AIMessage를 이전 대화에 포함시키기
# ---------------------------------------------------
# 대화 이력을 유지하려면 이전 AIMessage를 메시지 리스트에 포함시켜요
from langchain.messages import SystemMessage, HumanMessage, AIMessage

# 3번의 대화 이력이 있는 상황을 구성해요
conversation_history = [
    SystemMessage("당신은 친절한 어시스턴트입니다."),
    HumanMessage("대한민국의 수도는?"),
    AIMessage("대한민국의 수도는 서울입니다."),  # 이전 응답을 포함시켜요
    HumanMessage("그 도시의 인구는?"),           # 이전 문맥을 참고해야 해요
]

# 모델이 이전 대화에서 "서울"임을 파악해서 답변해요
response = model.invoke(conversation_history)
print(f"응답: {response.content}")
print()
# 모델이 이전 대화 맥락을 기억하여 서울의 인구를 답변했어요!

응답: 서울의 인구는 2023년 기준으로 약 9백만 명 이상입니다. 하지만 정확한 인구는 변화할 수 있으므로 최근의 통계를 확인하는 것이 좋습니다.



## 3. ToolMessage — 도구 호출 흐름

`ToolMessage`는 도구 실행 결과를 모델에 다시 전달하는 특별한 메시지예요. LangGraph 에이전트의 ReAct 루프에서 핵심 역할을 해요.

도구 호출 전체 흐름은 아래와 같이 4단계로 이루어져요:

```mermaid
sequenceDiagram
    participant U as 사용자
    participant M as LLM 모델
    participant T as 도구 실행

    U->>M: HumanMessage (서울 날씨 알려줘)
    M-->>T: AIMessage (tool_calls: get_weather, id=call_123)
    T-->>M: ToolMessage (content: 맑음 10도, tool_call_id=call_123)
    M-->>U: AIMessage (content: 서울은 현재 맑고 10도입니다)
```

### AIMessage의 두 가지 모드

| 모드 | `content` | `tool_calls` | 의미 |
|------|-----------|-------------|------|
| 텍스트 응답 | `"서울의 수도는..."` | `[]` (빈 리스트) | LLM이 직접 답변 |
| 도구 호출 | `""` (빈 문자열) | `[{name, args, id}]` | LLM이 도구 사용을 요청 |


In [11]:
from langchain.messages import AIMessage, HumanMessage, ToolMessage

# 1단계 : 모델이 도구 호출을 포함한 AIMessage 생성
#  원래는 LangGraph가 알아서 자동으로 생성.
ai_with_tool_call = AIMessage(
    content = "",
    tool_calls=[
        {
            "name": "get_weather",
            "args": {"location": "서울"},
            "id": "call_abc123" # tool의 id.
        }
    ]
)

print(ai_with_tool_call.tool_calls)

[{'name': 'get_weather', 'args': {'location': '서울'}, 'id': 'call_abc123', 'type': 'tool_call'}]


In [12]:
# 2단계 : 도구 실행 결과를 ToolMessage로 감싸주기
weather_result = "날씨: 맑음, 기온: 10도C, 습도: 45%" # 도구 실행 결과.

# ToolMessage 생성
#  tool_call_id를 지정 -> AIMessage의 tool_calls[0]['id']와 반드시 일치.
tool_msg = ToolMessage(
    content=weather_result,
    tool_call_id="call_abc123"
)

In [13]:
# 3단계 : 전체 도구 호출 대화 흐름
#  HumanMessage -> AIMessage(tool_calls) -> ToolMessage -> AIMessage(응답)
full_conversation = [
    HumanMessage("서울의 현재 날씨 알려줘."), # 1단계 : 사용자 질문
    ai_with_tool_call,                  # 2단계 : 모델이 도구 호출을 결정(get_weather 도구를 사용하기로 결정)
    tool_msg                            # 3단계 : 도구 실행 결과 전달.
]

final_response = model.invoke(full_conversation)
print(final_response.content)

현재 서울의 날씨는 맑고, 기온은 10도C, 습도는 45%입니다.


### 실제 도구 바인딩과 자동 도구 호출 확인

수동으로 메시지를 구성하는 방법을 이해했어요. 이제 실제로 모델에 도구를 바인딩하고, 모델이 자동으로 도구를 선택하는 과정을 확인해볼게요.


In [ ]:
# ---------------------------------------------------
# 실제 도구 바인딩 후 AIMessage 구조 확인
# ---------------------------------------------------
from pydantic import BaseModel, Field
from langchain.messages import HumanMessage


# Pydantic으로 도구 스키마 정의
class GetWeather(BaseModel):
    """현재 날씨 정보를 가져오는 도구예요."""

    location: str = Field(description="도시명. 예: 서울, 부산")


class SearchNews(BaseModel):
    """뉴스를 검색하는 도구예요."""

    query: str = Field(description="검색어")
    max_results: int = Field(default=3, description="최대 결과 수")


# 모델에 도구를 바인딩해요
model_with_tools = model.bind_tools([GetWeather, SearchNews])

# 날씨 관련 질문: 모델이 GetWeather 도구를 선택해요
response_tool = model_with_tools.invoke([HumanMessage("오늘 부산 날씨 알려줘")])

# === 도구 호출 발생 시 AIMessage ===
print(f"content (비어 있어요): '{response_tool.content}'")
print()
# tool_calls 내용:
for tc in response_tool.tool_calls:
    print(f"  이름: {tc['name']}")
    print(f"  인자: {tc['args']}")
    print(f"  ID:  {tc['id']}")
    #   (이 ID를 ToolMessage의 tool_call_id에 사용해야 해요)

content (비어 있어요): ''

  이름: GetWeather
  인자: {'location': '부산'}
  ID:  call_d6mmrCgxK3fF2RbkQpsFUJqU


In [20]:
# ---------------------------------------------------
# tool_call_id 매칭으로 완전한 ReAct 루프 구성
# ---------------------------------------------------
# 실제 도구를 실행하는 함수 (여기서는 더미 데이터 반환)
def execute_get_weather(location: str) -> str:
    """날씨 API를 호출하는 더미 함수예요."""
    weather_db = {
        "서울": "맑음, 15도C",
        "부산": "흐림, 18도C",
        "제주": "비, 20도C",
    }
    return weather_db.get(location, f"{location}: 데이터 없음")

# tool_call_id를 AIMessage에서 추출
tool_call = response_tool.tool_calls[0]
location_arg = tool_call['args']['location']
call_id = tool_call['id']

# 도구 실행 및 ToolMessage 생성.
weather_data = execute_get_weather(location_arg) # 도구 호출의 결과. ToolMessage의 content
auto_tool_msg = ToolMessage(
    content = weather_data,
    tool_call_id=call_id
)

original_question = HumanMessage("오늘 부산 날씨 알려줘")
messages = [
    original_question,
    response_tool,
    auto_tool_msg
]

final_answer = model_with_tools.invoke(messages)
final_answer

AIMessage(content='오늘 부산의 날씨는 흐리고 기온은 18도입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 137, 'total_tokens': 154, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a5e4ad44f7', 'id': 'chatcmpl-DecRQ9Vm4Au0invDu6MKf3UgRl8Zo', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e1b3a-3326-7632-b5dc-664ad83c73e5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 137, 'output_tokens': 17, 'total_tokens': 154, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## 4. 메시지 콘텐츠 형식 — 텍스트부터 멀티모달까지

메시지의 `content`는 단순 문자열부터 복잡한 멀티모달 블록까지 다양한 형식을 지원해요. LangChain V1에서는 두 가지 멀티모달 형식을 제공해요.

| 형식 | 특징 | 사용 시점 |
|------|------|----------|
| **문자열** | 단순 텍스트 | 기본 대화, 텍스트 처리 |
| **Provider 네이티브** | `image_url` 형식 | OpenAI, Anthropic 특정 기능 사용 시 |
| **표준 content_blocks** | `url` 형식 (V1 신기능) | 모든 제공자에서 동일하게 동작 |


In [ ]:
from langchain.messages import HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 콘텐츠 형식 1: 문자열 (가장 기본)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
import urllib.request
import base64

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 콘텐츠 형식 2: Provider 네이티브 (image_url 형식)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 콘텐츠 형식 3: 표준 content_blocks (V1 신기능 - 권장)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 멀티모달 메시지로 모델 호출 (gpt-4o-mini는 비전 지원)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 표준 content_blocks로 동일한 결과 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. 메시지 유틸리티 함수

LangGraph에서 자주 사용하는 메시지 처리 패턴들을 배워봐요. `pretty_print()`는 디버깅 시 메시지 구조를 빠르게 확인할 때 유용해요.


In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: pretty_print(): 메시지를 보기 좋게 출력해요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain.messages import ToolMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 메시지 정보 출력 헬퍼 함수
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. 실습: 멀티 에이전트 대화 구성

지금까지 배운 개념을 종합해서 직접 대화를 구성해봐요. `name` 필드를 사용하여 두 에이전트가 주고받는 대화를 만들어보세요.


In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

# ============================================================
# TODO: 아래 코드를 완성해서 두 에이전트 대화를 구성해보세요
# 힌트: name 필드를 사용해서 "researcher"와 "analyst" 에이전트를 구분해요
#       각 에이전트는 HumanMessage 혹은 AIMessage에 name을 지정해요
# 예상 결과: 두 에이전트 간의 자연스러운 대화 흐름이 출력되어야 해요
# ============================================================


# TODO: 아래 대화를 완성해보세요
# researcher 에이전트가 분석 결과를 전달하고
# analyst 에이전트가 해석을 제공하는 시나리오예요


    # TODO: researcher 에이전트의 메시지를 추가해보세요
    # 힌트: HumanMessage(content="...", name="researcher")
    # HumanMessage(...),

    # TODO: analyst 에이전트의 응답을 추가해보세요
    # 힌트: AIMessage(content="...", name="analyst")
    # AIMessage(...),

    # TODO: 새로운 질문을 추가해보세요
    # HumanMessage(...),


# TODO: 대화를 모델에 전달하고 결과를 출력해보세요
# response = model.invoke(multi_agent_conversation)
# print(response.content)

# TODO: 위의 주석을 해제하고 코드를 완성해보세요!
# name 필드로 에이전트를 구분하는 패턴은 Multi-Agent 파트에서 자주 사용돼요.


## 7. 메시지 타입 확인과 대화 이력 관리

실무에서는 메시지 타입을 확인하거나 필터링해야 할 경우가 자주 생겨요. 특히 LangGraph 상태(State)에서 메시지를 처리할 때 유용한 패턴들을 알아봐요.


In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 메시지 타입 확인 패턴
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 대화 이력 요약 패턴
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **4가지 메시지 타입**: `SystemMessage`(모델 지침), `HumanMessage`(사용자 입력), `AIMessage`(모델 응답), `ToolMessage`(도구 결과) 각각의 역할과 사용 시점
- **메시지 메타데이터**: `name`으로 발신자를 구별하고, `id`로 메시지를 고유 추적해요. LangGraph의 `add_messages` reducer와 연계돼요
- **딕셔너리 형식**: `{"role": "user", "content": "..."}` 형식으로 간결하게 메시지를 구성해요. 역할 이름은 `system`, `user`, `assistant`, `tool`을 사용해요
- **ToolMessage ID 매칭**: `tool_call_id`는 반드시 `AIMessage`의 `tool_calls[i]['id']`와 일치해야 해요. ID 불일치는 가장 흔한 도구 호출 오류예요
- **멀티모달 콘텐츠**: Provider 네이티브(`image_url`) 방식과 표준 `content_blocks`(`url`) 방식 두 가지를 지원해요. 새 코드에서는 표준 방식을 권장해요
- **V1 import 경로**: `from langchain.messages import ...` (langchain_core 대신 langchain 사용)


## 다음 노트북 예고

다음 `04-StateGraph-Basics.ipynb`에서는 **StateGraph, State, Node, Edge, START/END**를 배워요. 이번 노트북에서 배운 메시지 타입들이 LangGraph의 상태(State)에 어떻게 저장되고 관리되는지, `add_messages` reducer가 메시지 ID를 어떻게 활용하는지 직접 확인할 수 있어요.
